In [11]:
import pandas as pd

sentences = pd.read_csv("./label_train.csv")
sentences['semantic_level'] = sentences['semantic_level'].apply(lambda x: [f"L{x}"])
sentences

,semantic_level,sentence
0,[L1],A bar chart entitled “Mean of Temperature Maxi...
1,[L1],The Mean of temp_max is plotted on the vertica...
2,[L1],The weather type is plotted on the horizontal ...
3,[L1],There are 5 bars corresponding to 5 precipitat...
4,[L1],The bars are solid blue.
...,...,...
2142,[L2],The continent with the highest amount of both ...
2143,[L2],Bolivia is the country with the lowest amount ...
2144,[L3],"A big part of Africa doesn't have many tests, ..."
2145,[L3],"In most cases, there is a strong positive corr..."


In [12]:
import json

with open('./keywords_dict.json') as f:
    keywords_dict = json.load(f)

In [13]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from collections import Counter

# Download required NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

# 1) Load the lemmatizer
lemmatizer = WordNetLemmatizer()

# 2) Precompute lemmatized keyword sets
keyword_lemma_dict = {}
for category, keywords in keywords_dict.items():
    lemma_set = set()
    for kw in keywords:
        tokens = word_tokenize(kw.lower().strip())
        if tokens:
            lemma = lemmatizer.lemmatize(tokens[0])
            lemma_set.add(lemma)
    keyword_lemma_dict[category] = lemma_set

category_list = list(keyword_lemma_dict.keys())

# 3) New compute function: input is one sentence → one vector
def compute_category_features(sentence, keyword_lemma_dict, category_list):
    tokens = word_tokenize(sentence.lower())
    lemmas = [lemmatizer.lemmatize(tok) for tok in tokens if tok.isalpha()]
    total = max(1, len(lemmas))
    counts = Counter()
    for lm in lemmas:
        for cat in category_list:
            if lm in keyword_lemma_dict[cat]:
                counts[cat] += 1
    return [counts[cat] / total for cat in category_list]

# 4) Apply to your DataFrame
sentences["kw_feature_vector"] = (
    sentences["sentence"]
    .apply(lambda s: compute_category_features(s, keyword_lemma_dict, category_list))
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\niiv_student\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\niiv_student\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\niiv_student\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


# Splitting for train and test

In [14]:
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit


split = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)

for train_index, sample_index in split.split(sentences, sentences['semantic_level']):
    test_sentences = sentences.iloc[sample_index]


train_sentences = sentences.drop(test_sentences.index)


print(train_sentences.shape)
print(test_sentences.shape)

(1932, 3)
(215, 3)


In [15]:
import torch
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode_dataset(texts, kw_freqs):
    tok = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    return {
        "input_ids":      tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "kw_freq":        torch.tensor(kw_freqs, dtype=torch.float)
    }

# Example usage:
texts    = train_sentences["sentence"].tolist()
kw_freqs = train_sentences["kw_feature_vector"].tolist()  # make sure this column is a list-per-row
batch = encode_dataset(texts, kw_freqs)

input_ids_tensor      = batch["input_ids"]
attention_mask_tensor = batch["attention_mask"]
kw_freq_tensor = batch["kw_freq"]


In [16]:
from sklearn.preprocessing import MultiLabelBinarizer
import torch

mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(train_sentences["semantic_level"])
labels_tensor  = torch.tensor(labels, dtype=torch.float)

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel

class BiLSTMWithBERT(nn.Module):
    def __init__(self, hidden_dim, num_labels, num_categories, unfreeze_last=2):
        super(BiLSTMWithBERT, self).__init__()
        # Load BERT
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        # Freeze all but last `unfreeze_last` layers + pooler
        for name, param in self.bert.named_parameters():
            param.requires_grad = False
            for layer_idx in range(12 - unfreeze_last, 12):
                if f"encoder.layer.{layer_idx}" in name or "pooler" in name:
                    param.requires_grad = True

        self.lstm    = nn.LSTM(
            input_size  = 768,
            hidden_size = hidden_dim,
            bidirectional= True,
            batch_first = True
        )
        self.dropout_text = nn.Dropout(0.3)

        self.fc_kw1    = nn.Linear(num_categories, 32)
        self.dropout_kw = nn.Dropout(0.3)
        self.fc_kw2    = nn.Linear(32, 16)

        concat_dim = hidden_dim * 2 + 16
        self.fc_out = nn.Linear(concat_dim, num_labels)

    def forward(self, input_ids, attention_mask, kw_freq):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = bert_out.last_hidden_state         

        lstm_out, _ = self.lstm(x)               
        pooled = torch.mean(lstm_out, dim=1)     
        pooled = self.dropout_text(pooled)

        k = F.relu(self.fc_kw1(kw_freq))         
        k = self.dropout_kw(k)
        k = F.relu(self.fc_kw2(k))        

        combined = torch.cat([pooled, k], dim=1)  
        logits   = self.fc_out(combined)        
        return logits

In [18]:
import numpy as np

class EarlyStopping:
    def __init__(self, patience=6, min_delta=0.0, path="checkpoint.pth"):
        self.patience  = patience
        self.min_delta = min_delta
        self.best_loss = np.inf
        self.counter   = 0
        self.path      = path

    def __cassll__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter > self.patience:
                return True 
        return False

In [19]:
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import StepLR



dataset = TensorDataset(input_ids_tensor, attention_mask_tensor, labels_tensor, kw_freq_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

hidden_dim = 128
output_dim = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMWithBERT(hidden_dim, output_dim, len(keywords_dict.keys())).to(device)


class_counts = train_sentences["semantic_level"].value_counts().sort_index().tolist()
class_weights = torch.tensor([sum(class_counts) / c for c in class_counts], dtype=torch.float)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
scheduler = StepLR(optimizer, step_size=3, gamma=0.5)
early_stopper = EarlyStopping(patience=5, min_delta=1e-4,
                              path="best_model.pth")

for epoch in range(100):
    model.train()
    total_loss = 0
    for input_ids, attention_mask, label_batch, kw_freq in train_loader:
        input_ids, attention_mask, label_batch, kw_freq = (
            input_ids.to(device),
            attention_mask.to(device),
            label_batch.to(device),
            kw_freq.to(device)
        )
        optimizer.zero_grad()
        output = model(input_ids, attention_mask, kw_freq_tensor)
        loss = criterion(output, label_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}")

    if early_stopper(total_loss, model):
        print(f"Stopping early at epoch {epoch} (no improvement for {early_stopper.patience} epochs).")
        break


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 32 but got size 1932 for tensor number 1 in the list.

# Evaluation

In [ ]:
test_labels = mlb.transform(test_sentences["semantic_level"])
test_labels_tensor  = torch.tensor(test_labels, dtype=torch.float)

In [ ]:
from sklearn.metrics import (
    hamming_loss,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

texts_test    = test_sentences["sentence"].tolist()
kw_freqs_test = test_sentences["kw_feature_vector"].tolist() 
batch_test = encode_dataset(texts_test, kw_freqs_test)

test_input_ids_tensor      = batch_test["input_ids"]
test_attention_mask_tensor = batch_test["attention_mask"]
test_kw_freq_tensor = batch_test["kw_freq"]


dataset = TensorDataset(test_input_ids_tensor, test_attention_mask_tensor, test_labels_tensor, test_kw_freq_tensor)
val_loader = DataLoader(dataset, batch_size=32, shuffle=True)

model.eval()
probs_all = []

with torch.no_grad():
    for test_input_ids, test_attention_mask, test_label_batch, test_kw_freq in val_loader:
        test_input_ids, test_attention_mask, test_label_batch, test_kw_freq = (
            test_input_ids.to(device),
            test_attention_mask.to(device),
            test_label_batch.to(device),
            test_kw_freq.to(device)
        )

        test_logits = model(test_input_ids, test_attention_mask, test_kw_freq)
        test_probs  = torch.sigmoid(test_logits).cpu().numpy()
        probs_all.extend(test_probs)

y_true = test_labels_tensor.numpy()

In [ ]:
from sklearn.metrics import (
    hamming_loss,
    classification_report
)


y_true = test_labels_tensor.numpy()
y_pred = (np.array(probs_all) >= threshold).astype(int)

ham_loss = hamming_loss(y_true, y_pred)
print(f"Hamming loss: {ham_loss:.4f}")
print(f"Hamming accuracy: {1 - ham_loss:.4f}")

target_names = [str(c) for c in mlb.classes_]
print("\nClassification report (per label):")
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=[str(c) for c in mlb.classes_],
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report_dict).T

df_plot = df_report.drop(index=['micro avg', 'macro avg', 'weighted avg', 'samples avg'])

plt.figure(figsize=(8, 6))
sns.heatmap(
    df_plot.iloc[:, :3],
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar_kws={'label': 'Score'}
)
plt.title("Per-Label Precision / Recall / F1-Score")
plt.ylabel("Labels")
plt.xlabel("Metrics")
plt.tight_layout()
plt.show()

print(df_report.T[['micro avg', 'macro avg', 'weighted avg', 'samples avg']])

In [ ]:
checkpoint = {
    "hidden_dim":         hidden_dim,
    "num_labels":         output_dim,
    "num_categories":     len(keywords.keys()),
    "unfreeze_last":      2,
    "mlb_classes":        mlb.classes_.tolist(),
    "tokenizer_name":     "bert-base-uncased",
    "model_state":        model.state_dict(),
    "optimizer_state":    optimizer.state_dict(),
    "scheduler_state":    scheduler.state_dict(),
    "epoch":              epoch,
}
torch.save(checkpoint, './bilstm_kwfreq.pth')
tokenizer.save_pretrained("./bilstm_kwfreq/checkpoint/tokenizer/")

ckpt = torch.load("bilstm_bert_checkpoint.pth", map_location="cpu")

# 1) Rebuild the tokenizer
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("checkpoint/tokenizer/")

# 2) Recreate your MultiLabelBinarizer
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
mlb.classes_ = ckpt["mlb_classes"]

# 3) Reconstruct the model
model = BiLSTMWithBERT(
    hidden_dim    = ckpt["hidden_dim"],
    num_labels    = ckpt["num_labels"],
    unfreeze_last = ckpt["unfreeze_last"]
)
model.load_state_dict(ckpt["model_state"])

# 4) (If you want to resume training) recreate optimizer & scheduler, then load their states
optimizer.load_state_dict(ckpt["optimizer_state"])
scheduler.load_state_dict(ckpt["scheduler_state"])

# 5) Move to device & set eval/train as needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
